In [1]:
from pathlib import Path
import sys

# Ensure project root is on sys.path for imports
project_root = Path.cwd()
while not (project_root / "main.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dataloader import load_illness_data

df = load_illness_data("SCZ", in_notebook=True)

Loading data for illness SCZ at /Users/leonackermann/Library/CloudStorage/GoogleDrive-leonmax.ackermann@googlemail.com/My Drive/Uni/Master/4/MasterThesis/ml-genetics4psychiatry/data/tmpDATA-Leon/donnees_MRI_SCZ_only_variants_clumping_p_thr_0.0001all.txt


In [2]:
from dataloader import preprocess

X_train, y_train, X_test, y_test = preprocess(df=df, target="Z_scores_SCZ", testsize = 0.2)


In [17]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

In [18]:
# Create spline regression pipeline
spline_reg = Pipeline([
    ("spline", SplineTransformer(n_knots=3, degree=3, include_bias=True)),
    ("linear", Lasso())
])

# Train the model
spline_reg.fit(X_train, y_train)

# Make predictions
y_pred_train = spline_reg.predict(X_train)
y_pred_test = spline_reg.predict(X_test)

# Calculate performance metrics
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

# Perform cross-validation
cv_scores = cross_val_score(
    spline_reg, X_train, y_train, cv=5, scoring="neg_mean_squared_error"
)
cv_mse = -cv_scores.mean()
cv_std = cv_scores.std()

In [19]:
# print results
print(f"Train MSE: {train_mse:.4f}, Train R2: {train_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}, Test R2: {test_r2:.4f}")
print(f"Cross-Validation MSE: {cv_mse:.4f} ± {cv_std:.4f}") 

Train MSE: 22.0255, Train R2: 0.0000
Test MSE: 22.2532, Test R2: -0.0040
Cross-Validation MSE: 22.0526 ± 0.3722
